In [ ]:
!pip install torch torchvision torchaudio
!pip install transformers
!pip install datasets
!pip install peft
!pip install evaluate
!pip install accelerate
!pip install bitsandbytes
!pip install trl
!pip install pandas numpy matplotlib seaborn
!pip install scikit-learn tqdm
!pip install huggingface_hub

In [ ]:
import os
import json
import logging
import re
import glob
import numpy as np
import pandas as pd
import torch
import gc
from pickle import UnpicklingError
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from datasets import Dataset, disable_caching, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import warnings
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Disable datasets caching
disable_caching()

print("Libraries imported successfully!")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
CONFIG = {
    # Model Configuration
    'MODEL_NAME': "google/gemma-2-2b-it",
    'MAX_LENGTH': 2048,
    'MAX_DOCS': 5000,
    
    # Training Configuration
    'LEARNING_RATE': 2e-4,
    'NUM_EPOCHS': 3,
    'TRAIN_BATCH_SIZE': 2,
    'EVAL_BATCH_SIZE': 2,
    'GRADIENT_ACCUMULATION_STEPS': 8,
    'WARMUP_STEPS': 100,
    'EVAL_STEPS': 200,
    'SAVE_STEPS': 200,
    'LOGGING_STEPS': 50,
    
    # LoRA Configuration
    'LORA_R': 16,
    'LORA_ALPHA': 32,
    'LORA_DROPOUT': 0.1,
}

# Paths Setup
PATHS = {
    'BASE_DIR': "/kaggle/working",
    'LOCAL_TEXT_DIR': "/kaggle/input/legal-texts",
    'KAGGLE_DATASET_PATH': "/kaggle/input/llm-fine-tuning-dataset-of-indian-legal-texts/legal_qa_dataset.json",
    'MAPPING_FILE': "/kaggle/input/legal-mappings/legal_code_mapping.json",
}

# Create output directories
OUTPUT_DIRS = {
    'RESULTS_DIR': os.path.join(PATHS['BASE_DIR'], "results"),
    'LOGS_DIR': os.path.join(PATHS['BASE_DIR'], "logs"),
    'FINAL_DIR': os.path.join(PATHS['BASE_DIR'], "gemma_legal_qna_final"),
    'ERROR_DIR': os.path.join(PATHS['BASE_DIR'], "model_on_error"),
    'CACHE_DIR': os.path.join(PATHS['BASE_DIR'], "local_model_cache"),
    'DATA_DIR': os.path.join(PATHS['BASE_DIR'], "processed_data")
}

for directory in OUTPUT_DIRS.values():
    os.makedirs(directory, exist_ok=True)

print("Configuration and paths setup complete!")
print(f"Working Directory: {PATHS['BASE_DIR']}")
print(f"Model: {CONFIG['MODEL_NAME']}")

In [ ]:
class LegalTextProcessor:
    """Process various legal text sources into QnA format"""
    
    def __init__(self, local_text_dir: str, kaggle_dataset_path: str, mapping_file: str):
        self.local_text_dir = local_text_dir
        self.kaggle_dataset_path = kaggle_dataset_path
        self.mapping_file = mapping_file
        self.qa_pairs = []
        
    def load_local_texts(self) -> Dict[str, str]:
        """Load local legal text files (Constitution, BNSS, BNS, CrPC)"""
        texts = {}
        
        if not os.path.exists(self.local_text_dir):
            logger.warning(f"Local text directory not found: {self.local_text_dir}")
            return texts
            
        text_files = glob.glob(os.path.join(self.local_text_dir, "*.txt"))
        logger.info(f"Found {len(text_files)} text files")
        
        for file_path in tqdm(text_files, desc="Loading local texts"):
            filename = os.path.basename(file_path).lower()
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read().strip()
                    if content:
                        if 'constitution' in filename:
                            texts['constitution'] = content
                        elif 'bnss' in filename:
                            texts['bnss'] = content
                        elif 'bns' in filename:
                            texts['bns'] = content
                        elif 'crpc' in filename:
                            texts['crpc'] = content
                        else:
                            doc_name = os.path.splitext(filename)[0]
                            texts[doc_name] = content
                            
                logger.info(f"Loaded {filename}: {len(content):,} characters")
            except Exception as e:
                logger.warning(f"Could not read {file_path}: {e}")
                
        return texts
    
    def load_mapping_data(self) -> Dict:
        """Load mapping between old and new legal codes"""
        try:
            if os.path.exists(self.mapping_file):
                with open(self.mapping_file, 'r', encoding='utf-8') as f:
                    mapping_data = json.load(f)
                logger.info(f"Loaded mapping file: {len(mapping_data)} entries")
                return mapping_data
            else:
                logger.warning(f"Mapping file not found: {self.mapping_file}")
        except Exception as e:
            logger.warning(f"Could not load mapping file: {e}")
        return {}
    
    def load_kaggle_dataset(self) -> List[Dict]:
        """Load Kaggle legal QnA dataset"""
        qa_data = []
        try:
            if os.path.exists(self.kaggle_dataset_path):
                if self.kaggle_dataset_path.endswith('.json'):
                    with open(self.kaggle_dataset_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if isinstance(data, list):
                            qa_data = data
                        elif isinstance(data, dict) and 'data' in data:
                            qa_data = data['data']
                elif self.kaggle_dataset_path.endswith('.jsonl'):
                    with open(self.kaggle_dataset_path, 'r', encoding='utf-8') as f:
                        for line in f:
                            qa_data.append(json.loads(line.strip()))
                            
                logger.info(f"Loaded Kaggle dataset: {len(qa_data)} QnA pairs")
            else:
                logger.warning(f"Kaggle dataset not found: {self.kaggle_dataset_path}")
        except Exception as e:
            logger.warning(f"Could not load Kaggle dataset: {e}")
            
        return qa_data
    
    def create_qa_from_text(self, text: str, doc_type: str, chunk_size: int = 1000) -> List[Dict]:
        """Generate QnA pairs from legal text using rule-based approach"""
        qa_pairs = []
        
        if doc_type == 'constitution':
            # Constitution articles
            articles = re.findall(r'(Article \d+[^A]*?)(?=Article \d+|$)', text, re.IGNORECASE | re.DOTALL)
            for article in articles:
                article = article.strip()
                if len(article) > 50:
                    match = re.search(r'Article (\d+)', article, re.IGNORECASE)
                    if match:
                        art_num = match.group(1)
                        question = f"What does Article {art_num} of the Indian Constitution state?"
                        answer = article[:500] + "..." if len(article) > 500 else article
                        qa_pairs.append({
                            'question': question,
                            'answer': answer,
                            'source': f'Constitution Article {art_num}',
                            'doc_type': doc_type
                        })
        
        elif doc_type in ['bnss', 'bns', 'crpc']:
            # Legal code sections
            sections = re.findall(r'(Section \d+[^S]*?)(?=Section \d+|$)', text, re.IGNORECASE | re.DOTALL)
            for section in sections:
                section = section.strip()
                if len(section) > 50:
                    match = re.search(r'Section (\d+)', section, re.IGNORECASE)
                    if match:
                        sec_num = match.group(1)
                        question = f"What is stated in Section {sec_num} of {doc_type.upper()}?"
                        answer = section[:500] + "..." if len(section) > 500 else section
                        qa_pairs.append({
                            'question': question,
                            'answer': answer,
                            'source': f'{doc_type.upper()} Section {sec_num}',
                            'doc_type': doc_type
                        })
        
        else:
            # Generic legal text - create chunks
            chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
            for i, chunk in enumerate(chunks):
                if len(chunk.strip()) > 100:
                    question = f"What legal provisions are mentioned in {doc_type}?"
                    answer = chunk.strip()
                    qa_pairs.append({
                        'question': question,
                        'answer': answer,
                        'source': f'{doc_type} - Part {i+1}',
                        'doc_type': doc_type
                    })
        
        return qa_pairs
    
    def create_mapping_qa(self, mapping_data: Dict) -> List[Dict]:
        """Generate QnA pairs from legal code mappings"""
        qa_pairs = []
        
        for old_provision, new_provision in mapping_data.items():
            if isinstance(new_provision, dict):
                new_section = new_provision.get('section', '')
                new_description = new_provision.get('description', '')
                
                # Create mapping question
                question = f"What is the equivalent of {old_provision} in the new legal codes?"
                answer = f"{old_provision} corresponds to {new_section}. {new_description}"
                
                qa_pairs.append({
                    'question': question,
                    'answer': answer,
                    'source': 'Legal Code Mapping',
                    'doc_type': 'mapping'
                })
                
                # Create reverse mapping question
                if new_section:
                    question = f"Which old provision does {new_section} replace?"
                    answer = f"{new_section} replaces {old_provision}. {new_description}"
                    
                    qa_pairs.append({
                        'question': question,
                        'answer': answer,
                        'source': 'Legal Code Mapping',
                        'doc_type': 'mapping'
                    })
        
        return qa_pairs
    
    def process_all_sources(self) -> List[Dict]:
        """Process all data sources and create comprehensive QnA dataset"""
        all_qa_pairs = []
        
        # 1. Process local legal texts
        logger.info("Processing local legal text files...")
        local_texts = self.load_local_texts()
        for doc_type, text in local_texts.items():
            qa_pairs = self.create_qa_from_text(text, doc_type)
            all_qa_pairs.extend(qa_pairs)
            logger.info(f"Generated {len(qa_pairs)} QnA pairs from {doc_type}")
        
        # 2. Process Kaggle dataset
        logger.info("Processing Kaggle dataset...")
        kaggle_data = self.load_kaggle_dataset()
        for item in kaggle_data:
            if 'question' in item and 'answer' in item:
                all_qa_pairs.append({
                    'question': item['question'],
                    'answer': item['answer'],
                    'source': item.get('source', 'Kaggle Dataset'),
                    'doc_type': 'kaggle'
                })
            elif 'instruction' in item and 'output' in item:
                all_qa_pairs.append({
                    'question': item['instruction'],
                    'answer': item['output'],
                    'source': item.get('source', 'Kaggle Dataset'),
                    'doc_type': 'kaggle'
                })
        
        logger.info(f"Added {len(kaggle_data)} QnA pairs from Kaggle dataset")
        
        # 3. Process mapping data
        logger.info("Processing legal code mappings...")
        mapping_data = self.load_mapping_data()
        mapping_qa = self.create_mapping_qa(mapping_data)
        all_qa_pairs.extend(mapping_qa)
        logger.info(f"Generated {len(mapping_qa)} QnA pairs from mappings")
        
        logger.info(f"Total QnA pairs created: {len(all_qa_pairs)}")
        return all_qa_pairs

print("Data processing classes defined!")

In [ ]:
# Initialize data processor
processor = LegalTextProcessor(
    local_text_dir=PATHS['LOCAL_TEXT_DIR'],
    kaggle_dataset_path=PATHS['KAGGLE_DATASET_PATH'],
    mapping_file=PATHS['MAPPING_FILE']
)

# Process all data sources
print("Starting data processing...")
qa_pairs = processor.process_all_sources()

# Limit dataset size if needed
if len(qa_pairs) > CONFIG['MAX_DOCS']:
    qa_pairs = qa_pairs[:CONFIG['MAX_DOCS']]
    logger.info(f"Limited dataset to {CONFIG['MAX_DOCS']} QnA pairs")

# Save processed data
processed_data_path = os.path.join(OUTPUT_DIRS['DATA_DIR'], 'processed_qa_pairs.json')
with open(processed_data_path, 'w', encoding='utf-8') as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print(f"Saved processed data to: {processed_data_path}")
print(f"Total QnA pairs: {len(qa_pairs)}")

In [ ]:
# Analyze the dataset
df = pd.DataFrame(qa_pairs)

print("DATASET ANALYSIS")
print("=" * 50)
print(f"Total samples: {len(df)}")
print(f"Unique sources: {df['source'].nunique()}")
print(f"Document types: {df['doc_type'].value_counts().to_dict()}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribution by document type
df['doc_type'].value_counts().plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Distribution by Document Type')
axes[0,0].set_xlabel('Document Type')
axes[0,0].set_ylabel('Count')

# 2. Question length distribution
question_lengths = [len(q.split()) for q in df['question']]
axes[0,1].hist(question_lengths, bins=30, color='lightgreen', alpha=0.7)
axes[0,1].set_title('Question Length Distribution')
axes[0,1].set_xlabel('Number of Words')
axes[0,1].set_ylabel('Frequency')

# 3. Answer length distribution
answer_lengths = [len(a.split()) for a in df['answer']]
axes[1,0].hist(answer_lengths, bins=30, color='coral', alpha=0.7)
axes[1,0].set_title('Answer Length Distribution')
axes[1,0].set_xlabel('Number of Words')
axes[1,0].set_ylabel('Frequency')

# 4. Top sources
top_sources = df['source'].value_counts().head(10)
top_sources.plot(kind='barh', ax=axes[1,1], color='gold')
axes[1,1].set_title('Top 10 Sources')
axes[1,1].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f"Average question length: {np.mean(question_lengths):.1f} words")
print(f"Average answer length: {np.mean(answer_lengths):.1f} words")
print(f"Max question length: {max(question_lengths)} words")
print(f"Max answer length: {max(answer_lengths)} words")


In [ ]:
def setup_lora_config():
    """Setup LoRA configuration for efficient fine-tuning"""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=CONFIG['LORA_R'],
        lora_alpha=CONFIG['LORA_ALPHA'],
        lora_dropout=CONFIG['LORA_DROPOUT'],
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

# Load model and tokenizer
logger.info(f"Loading model and tokenizer: {CONFIG['MODEL_NAME']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['MODEL_NAME'],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Setup LoRA
logger.info("Setting up LoRA configuration...")
lora_config = setup_lora_config()
model = get_peft_model(model, lora_config)

print("Trainable Parameters:")
model.print_trainable_parameters()

# Save tokenizer and base model
tokenizer.save_pretrained(OUTPUT_DIRS['CACHE_DIR'])
model.save_pretrained(OUTPUT_DIRS['CACHE_DIR'])

print("Model and tokenizer setup complete!")

In [ ]:
def format_qa_for_training(qa_pairs: List[Dict], tokenizer, max_length: int = 2048) -> Dataset:
    """Format QnA pairs for causal language modeling training"""
    formatted_texts = []
    
    for qa in tqdm(qa_pairs, desc="Formatting QnA pairs"):
        # Format as instruction-following conversation
        if tokenizer.chat_template:
            # Use chat template if available
            messages = [
                {"role": "user", "content": qa['question']},
                {"role": "assistant", "content": qa['answer']}
            ]
            try:
                formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
            except:
                # Fallback formatting
                formatted_text = f"User: {qa['question']}\nAssistant: {qa['answer']}"
        else:
            # Simple instruction format
            formatted_text = f"### Human: {qa['question']}\n\n### Assistant: {qa['answer']}\n\n"
        
        formatted_texts.append(formatted_text)
    
    # Create dataset
    dataset = Dataset.from_dict({'text': formatted_texts})
    
    # Tokenize
    def tokenize_function(examples):
        tokenized = tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            padding=False,
            return_tensors=None
        )
        # For causal LM, labels are the same as input_ids
        tokenized['labels'] = tokenized['input_ids'].copy()
        return tokenized
    
    tokenized_dataset = dataset.map(
        tokenize_function, 
        batched=True, 
        remove_columns=['text'],
        desc="Tokenizing dataset"
    )
    
    return tokenized_dataset

# Format data for training
logger.info("Formatting data for training...")
train_dataset = format_qa_for_training(qa_pairs, tokenizer, CONFIG['MAX_LENGTH'])

# Split dataset
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset_split = train_dataset.select(range(train_size))
val_dataset_split = train_dataset.select(range(train_size, train_size + val_size))

logger.info(f"Training samples: {len(train_dataset_split)}")
logger.info(f"Validation samples: {len(val_dataset_split)}")

print("Data formatting and tokenization complete!")

In [ ]:
def find_latest_checkpoint(results_dir, error_dir):
    """Find the latest training checkpoint"""
    latest_ckpt, max_step = None, -1

    for ck in glob.glob(os.path.join(results_dir, "checkpoint-*")):
        m = re.search(r"checkpoint-(\d+)", ck)
        if not m:
            continue
        step = int(m.group(1))
        if step > max_step and any(
            os.path.exists(os.path.join(ck, fname))
            for fname in ["pytorch_model.bin", "adapter_model.bin", "adapter_model.safetensors"]
        ):
            max_step, latest_ckpt = step, ck

    if latest_ckpt is not None:
        return latest_ckpt

    if os.path.isdir(error_dir) and any(
        os.path.exists(os.path.join(error_dir, fname))
        for fname in ["pytorch_model.bin", "adapter_model.bin", "adapter_model.safetensors"]
    ):
        return error_dir

    return None

class TrainingProgressCallback(TrainerCallback):
    """Custom callback to track training progress"""
    
    def __init__(self):
        self.training_logs = []
    
    def on_log(self, args, state, control, model=None, logs=None, **kwargs):
        if logs:
            logs['step'] = state.global_step
            logs['epoch'] = state.epoch
            self.training_logs.append(logs.copy())

# Setup data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal language modeling
    pad_to_multiple_of=8
)

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIRS['RESULTS_DIR'],
    logging_dir=OUTPUT_DIRS['LOGS_DIR'],
    report_to="none",
    eval_strategy="steps",
    eval_steps=CONFIG['EVAL_STEPS'],
    save_strategy="steps",
    save_steps=CONFIG['SAVE_STEPS'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=CONFIG['LEARNING_RATE'],
    num_train_epochs=CONFIG['NUM_EPOCHS'],
    per_device_train_batch_size=CONFIG['TRAIN_BATCH_SIZE'],
    per_device_eval_batch_size=CONFIG['EVAL_BATCH_SIZE'],
    gradient_accumulation_steps=CONFIG['GRADIENT_ACCUMULATION_STEPS'],
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
    warmup_steps=CONFIG['WARMUP_STEPS'],
    logging_steps=CONFIG['LOGGING_STEPS'],
    dataloader_drop_last=True,
    remove_unused_columns=False,
    push_to_hub=False,
    hub_token=None
)

# Initialize progress callback
progress_callback = TrainingProgressCallback()

print("raining setup complete!")
print(f"Training Arguments:")
print(f"   - Learning Rate: {CONFIG['LEARNING_RATE']}")
print(f"   - Epochs: {CONFIG['NUM_EPOCHS']}")
print(f"   - Batch Size: {CONFIG['TRAIN_BATCH_SIZE']}")
print(f"   - Gradient Accumulation: {CONFIG['GRADIENT_ACCUMULATION_STEPS']}")

def compute_metrics(eval_pred):
    """Compute metrics for evaluation"""
    predictions, labels = eval_pred
    
    # Simple perplexity calculation
    # Note: This is a basic implementation for monitoring
    losses = []
    for pred, label in zip(predictions, labels):
        # Calculate loss for non-padded tokens
        mask = (label != -100)
        if mask.sum() > 0:
            # This is a simplified calculation
            losses.append(len(label[mask]))
    
    avg_length = np.mean(losses) if losses else 0
    
    return {
        'avg_pred_length': avg_length,
        'num_samples': len(predictions)
    }

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_split,
    eval_dataset=val_dataset_split,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[progress_callback]
)

print("Trainer initialized!")
print(f"Training Dataset Size: {len(train_dataset_split)}")
print(f"Validation Dataset Size: {len(val_dataset_split)}")

# Check for existing checkpoints
checkpoint = find_latest_checkpoint(OUTPUT_DIRS['RESULTS_DIR'], OUTPUT_DIRS['ERROR_DIR'])
if checkpoint:
    logger.info(f"Found existing checkpoint: {checkpoint}")
else:
    logger.info("Starting training from scratch")

# Start training
logger.info("Starting training...")
print("=" * 50)
print("TRAINING STARTED")
print("=" * 50)

try:
    # Clear cache before training
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    # Train the model
    train_result = trainer.train(resume_from_checkpoint=checkpoint)
    
    print("=" * 50)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 50)
    
    # Print training summary
    print(f"Training Summary:")
    print(f"   - Total Steps: {train_result.global_step}")
    print(f"   - Training Loss: {train_result.training_loss:.4f}")
    print(f"   - Training Time: {train_result.metrics['train_runtime']:.2f} seconds")
    
except Exception as e:
    logger.error(f"Training failed: {e}")
    print(f"Training Error: {e}")
    
    # Save model in error directory for recovery
    try:
        trainer.save_model(OUTPUT_DIRS['ERROR_DIR'])
        tokenizer.save_pretrained(OUTPUT_DIRS['ERROR_DIR'])
        logger.info(f"Model saved to error directory: {OUTPUT_DIRS['ERROR_DIR']}")
    except Exception as save_error:
        logger.error(f"Could not save model to error directory: {save_error}")

# Save final model
logger.info("Saving final model...")
try:
    trainer.save_model(OUTPUT_DIRS['FINAL_DIR'])
    tokenizer.save_pretrained(OUTPUT_DIRS['FINAL_DIR'])
    print(f"Final model saved to: {OUTPUT_DIRS['FINAL_DIR']}")
except Exception as e:
    logger.error(f"Could not save final model: {e}")

# Final evaluation
logger.info("Running final evaluation...")
try:
    eval_results = trainer.evaluate()
    print(f"Final Evaluation Results:")
    for key, value in eval_results.items():
        print(f"   - {key}: {value}")
        
    # Save evaluation results
    eval_results_path = os.path.join(OUTPUT_DIRS['FINAL_DIR'], 'eval_results.json')
    with open(eval_results_path, 'w') as f:
        json.dump(eval_results, f, indent=2)
        
except Exception as e:
    logger.error(f"Evaluation failed: {e}")